# 07 TeleOCR face aux modèles multimodaux, sur les mêmes bulletins



| Avantages | Inconvénients |
|---|---|
| Tourne **chez nous**, sur notre GPU : pas de service externe à faire homologuer | Rien ne montre qu'il lit l'**écriture manuscrite**, ni le **français** |
| Peut lire une **page entière** sans découpe, et repérer les zones  | Il **ne dit jamais « illisible »** : toutes ses erreurs sont silencieuses |


## La question

Sur **les mêmes bulletins** et **les mêmes images**, TeleOCR (local) lit-il aussi bien que les
modèles multimodaux du service : **gemma4**, **qwen3-vl** et **chandra-ocr** ?

## Le protocole

Tous les modèles reçoivent **exactement les mêmes images** et **les mêmes consignes** :

| Entrée | Ce que reçoit chaque modèle |
|---|---|
| **Ligne** | une ligne découpée (positions fixes) |
| **Bloc** | le bloc nom + prénom, avec ses libellés imprimés |
| **Page**  | la page entière, sans aucune découpe |



**Autonome :** le notebook charge lui-même les images et le Fiqual depuis S3, et appelle
lui-même les modèles. 

In [1]:
# 1. Reglages et donnees (charges ici, depuis S3)
import base64
import io
import json
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
from pathlib import Path

import mlflow
import pandas as pd
import requests
from PIL import Image
from scipy import stats

RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RACINE / "src"))
from mesure_qualite.evaluation import suivi
from mesure_qualite.perception import banc_htr as bh

TEST_RAPIDE = False          # True : 5 bulletins. False : N_BULLETINS.
N_BULLETINS = None            # None = tous les bulletins apparies (169 pour RG25)
ENTREES = ["ligne", "bloc"]   # ajouter "page" pour la page entiere (plus lent)
MODELES_SERVICE = ["gemma4-26b-moe", "qwen3-vl", "chandra-ocr-2"]   # les multimodaux du notebook 03
MODELES = ["teleocr"] + MODELES_SERVICE

BUCKET, CAMPAGNE = "projet-mesure-qualite-rp", "RG25"
FICHIER_FIQUAL = "RP_PMQ_FIQUAL2599.txt"
POS_UT, POS_CAB = 2, 4
COLS = {"NOM": 11, "PRENOM": 12}
ZONES = {"NOM": (0.155, 0.083, 0.245, 0.011), "PRENOM": (0.155, 0.094, 0.245, 0.011)}
BORDS_BLOC = (0.075, 0.072, 0.450, 0.105)
for cz in [RACINE / "configs" / f"positions_BI_{CAMPAGNE}.json",
           RACINE / "notebooks" / "configs" / f"positions_BI_{CAMPAGNE}.json"]:
    if cz.exists():
        ZONES = {k: tuple(v) for k, v in json.loads(cz.read_text(encoding="utf-8"))["zones"].items()
                 if k in ZONES}
        break
g, h, d_, b_ = BORDS_BLOC
ZONE_BLOC = (g, h, d_ - g, b_ - h)

RESULTATS = Path.home() / "work" / "banc-htr" / "resultats"
IMAGES = Path("/tmp/teleocr-vs-gemma4")              # images (videes au redemarrage, se refont)
for d in [RESULTATS, IMAGES / "page", IMAGES / "bloc", IMAGES / "ligne"]:
    d.mkdir(parents=True, exist_ok=True)
SUFFIXE = "_rapide" if TEST_RAPIDE else ""


def normaliser(t):
    """Identique au notebook 05."""
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return ""
    t = unicodedata.normalize("NFKD", str(t).strip())
    t = "".join(c for c in t if not unicodedata.combining(c)).upper()
    t = re.sub(r"[^A-Z0-9 ]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def decouper(image, zone):
    L, H = image.size
    x, y, w, h = zone
    return image.crop((int(x * L), int(y * H), int((x + w) * L), int((y + h) * H)))


import s3fs
fs = s3fs.S3FileSystem()
fichiers = [f for f in fs.find(f"{BUCKET}/data/raw/{CAMPAGNE}/BIE") if f.endswith("_BI.jpg")]
bi = pd.DataFrame({"chemin": fichiers})
m = bi.chemin.map(lambda c: Path(c).stem).str.split("_")
bi["cle"] = m.str[0] + "_" + m.str[1]
with fs.open(f"{BUCKET}/data/raw/{CAMPAGNE}/{FICHIER_FIQUAL}", "rb") as f:
    brut = pd.read_csv(f, sep="\x00", header=None, names=["ligne"], dtype=str,
                       encoding="latin-1", keep_default_na=False, engine="python")["ligne"]
ch = brut.str.split("|")
fq3 = pd.DataFrame(ch[ch.str[0] == "3"].tolist(), dtype=str)
fq3["cle"] = fq3[POS_UT].str.strip() + "_" + fq3[POS_CAB].str.strip()
ref = fq3[["cle", COLS["NOM"], COLS["PRENOM"]]].copy()
ref.columns = ["cle", "NOM", "PRENOM"]
APP = bi.merge(ref, on="cle", how="inner").drop_duplicates("cle").sort_values("cle").reset_index(drop=True)
n = 5 if TEST_RAPIDE else (N_BULLETINS or len(APP))
APP = APP.head(n)
REF = {(r.cle, c): normaliser(getattr(r, c)) for r in APP.itertuples() for c in ZONES}

for r in APP.itertuples():
    fp = IMAGES / "page" / f"{r.cle}.jpg"
    if not fp.exists():
        with fs.open(r.chemin, "rb") as src:
            Image.open(io.BytesIO(src.read())).convert("RGB").save(fp, quality=92)
    page = Image.open(fp).convert("RGB")
    if not (IMAGES / "bloc" / f"{r.cle}.png").exists():
        decouper(page, ZONE_BLOC).save(IMAGES / "bloc" / f"{r.cle}.png")
    for c, z in ZONES.items():
        fl = IMAGES / "ligne" / f"{r.cle}__{c}.png"
        if not fl.exists():
            decouper(page, z).save(fl)
print(f"{len(APP)} bulletins, {len(REF)} champs | mode {'TEST RAPIDE' if TEST_RAPIDE else 'COMPLET'}")

/home/onyxia/work/mesure-qualite-rp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


169 bulletins, 338 champs | mode COMPLET


In [5]:
# 2. Les deux modeles, avec les memes consignes que dans les notebooks precedents
import base64, io, json, os, re, shutil, subprocess, time
from pathlib import Path

import requests
from PIL import Image

# =====================================================================================
# Service (gemma4 etc.)
# =====================================================================================
BASE_URL = "https://llm.lab.sspcloud.fr/api"
if "OPENAI_API_KEY" not in os.environ and shutil.which("vault"):
    r = subprocess.run(["vault", "kv", "get", "-field=OPENAI_API_KEY", "onyxia-kv/farisazibert/LLM"],
                       capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        os.environ["OPENAI_API_KEY"] = r.stdout.strip()
if "OPENAI_API_KEY" not in os.environ:
    raise RuntimeError("Cle du service absente : la renseigner (Vault) avant de continuer.")
HEADERS = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}

PROMPT_BLOC = """Cette image montre un extrait de bulletin individuel du recensement francais.
Elle contient des lignes manuscrites precedees de leurs libelles imprimes :
"Nom :" et "Prenom :".

Transcris ce qui est ecrit a la main sur chacune, en MAJUSCULES, sans accent.

Regles :
- Ne corrige rien, n'ajoute rien, ne devine pas.
- Ne recopie pas les libelles imprimes, seulement l'ecriture manuscrite.
- Si une ligne n'est pas lisible avec certitude, mets exactement ILLISIBLE.

Reponds uniquement en JSON : {"NOM": "...", "PRENOM": "..."}"""
PROMPT_PAGE = PROMPT_BLOC.replace(
    "Cette image montre un extrait de bulletin individuel du recensement francais.\n"
    "Elle contient des lignes manuscrites precedees de leurs libelles imprimes :",
    "Cette image montre un bulletin individuel complet du recensement francais.\n"
    "Trouve les lignes manuscrites precedees des libelles imprimes :")
PROMPT_LIGNE = """Cette image montre une seule ligne manuscrite extraite d'un formulaire
administratif francais : un {champ} ecrit a la main, en majuscules d'imprimerie.

Transcris exactement ce qui est ecrit, en MAJUSCULES, sans accent.
Ne corrige rien, n'ajoute rien, ne devine pas.
Si l'ecriture n'est pas lisible avec certitude, reponds exactement : ILLISIBLE

Reponds uniquement par la transcription, sans phrase ni commentaire."""
NOM_CHAMP = {"NOM": "nom de famille", "PRENOM": "prenom"}


def _b64(image, cote_max=None):
    image = image.convert("RGB")
    if cote_max and max(image.size) > cote_max:
        image.thumbnail((cote_max, cote_max))
    t = io.BytesIO()
    image.save(t, format="PNG")
    return base64.b64encode(t.getvalue()).decode()


def service(modele, image, consigne, max_jetons=200, cote_max=None, tentatives=3):
    for essai in range(tentatives):
        try:
            r = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS, timeout=180, json={
                "model": modele, "temperature": 0, "max_tokens": max_jetons,
                "messages": [{"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{_b64(image, cote_max)}"}},
                    {"type": "text", "text": consigne}]}]})
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"].strip()
        except Exception:
            time.sleep(2 * (essai + 1))
    return ""


def json_champs(texte):
    m = re.search(r"\{.*\}", texte or "", re.S)
    try:
        d = json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        d = {}
    return {c: str(d.get(c, "") or "") for c in ZONES}


# =====================================================================================
# TeleOCR : environnement separe (transformers 4.x), revision figee, lecture par lots
# =====================================================================================
# --- Reglages de TeleOCR ----------------------------------------------------------------
ENTREES_TELE = ["ligne", "bloc"]   # retirer "page" pour aller beaucoup plus vite
N_TELE = None                              # nombre de bulletins lus par TeleOCR (None = tous)
JETONS = {"ligne": 64, "bloc": 256, "page": 768}
COTE_MAX = {"ligne": None, "bloc": None, "page": 1400}
INTERVALLE = 30                            # secondes entre deux affichages de progression

REVISION_TELEOCR = "e92585356c0d0b7b7a65938f3da035c6593cc9a6"     # code relu
CONSIGNE_TELEOCR = "Please output the text content from the image."
ENV_TELE = Path("/tmp/env-teleocr")
PY_TELE = ENV_TELE / "bin/python"
ENV_UV = {**os.environ, "UV_CACHE_DIR": "/tmp/uv-cache", "UV_LINK_MODE": "copy",
          "UV_PYTHON_INSTALL_DIR": "/tmp/uv-python"}


def env_tele_ok():
    return PY_TELE.exists() and subprocess.run([str(PY_TELE), "-c", "import torch, transformers"],
                                               capture_output=True).returncode == 0


if not env_tele_ok():
    print("installation de l'environnement TeleOCR (quelques minutes)...")
    for cmd in [f"rm -rf {ENV_TELE}",
                f"uv venv {ENV_TELE} --python 3.12 --python-preference only-managed",
                f"uv pip install --python {PY_TELE} torch torchvision 'transformers>=4.56,<5' accelerate pillow"]:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=ENV_UV)
        if r.returncode != 0:
            raise RuntimeError(r.stderr[-1000:])
assert env_tele_ok(), "environnement TeleOCR indisponible"

# Le script ECRIT CHAQUE LECTURE DES QU'ELLE EST FAITE (mode ajout) : rien n'est perdu en cas d'arret
SCRIPT_TELE_LOT = r'''
import json, sys, torch
from PIL import Image
from transformers import AutoProcessor, AutoModel
MAN, OUT, REV = sys.argv[1:4]
REPO = "XingChen-AGI/TeleOCR"
proc = AutoProcessor.from_pretrained(REPO, revision=REV, trust_remote_code=True, use_fast=True)
model, info = AutoModel.from_pretrained(REPO, revision=REV, trust_remote_code=True, torch_dtype=torch.bfloat16,
                                        attn_implementation="sdpa", output_loading_info=True)
model = model.cuda().eval()
manquants = [k for k in info["missing_keys"] if "lm_head" not in k]
if manquants or info["mismatched_keys"]:
    print("POIDS incomplets", manquants[:5], info["mismatched_keys"][:3], flush=True)
    sys.exit(2)
print("MODELE PRET", flush=True)
items = [json.loads(l) for l in open(MAN, encoding="utf-8")]
with open(OUT, "a", encoding="utf-8") as f, torch.no_grad():
    for it in items:
        try:
            im = Image.open(it["chemin"]).convert("RGB")
            if it.get("cote_max") and max(im.size) > it["cote_max"]:
                im.thumbnail((it["cote_max"], it["cote_max"]))
            msgs = [{"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": it["consigne"]}]}]
            chat = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            e = proc(text=[chat], images=[im], padding=True, return_tensors="pt").to(device=model.device, dtype=model.dtype)
            out = model.generate(**e, use_cache=True, max_new_tokens=it["max_tokens"], do_sample=False)
            txt = proc.batch_decode([out.cpu().tolist()[0][len(e.input_ids[0]):]], skip_special_tokens=True)[0].strip()
        except Exception as ex:                       # une image en echec n'arrete pas le lot
            print(f"ECHEC {it['id']} : {type(ex).__name__}: {ex}"[:300], flush=True)
            continue
        f.write(json.dumps({"id": it["id"], "texte": txt}, ensure_ascii=False) + "\n")
        f.flush()
'''
DOSSIER_TELE = Path.home() / "work/banc-htr/cache/teleocr"
DOSSIER_TELE.mkdir(exist_ok=True)
(DOSSIER_TELE / "lire_lot.py").write_text(SCRIPT_TELE_LOT, encoding="utf-8")

# Anciens caches (avant cette version) : reutilises seulement s'ils ont ete faits avec les memes reglages
ANCIENS_REGLAGES = {"ligne": (64, None), "bloc": (256, None), "page": (2048, 2000)}


def _lire_jsonl(chemin):
    if not chemin.exists():
        return {}
    res = {}
    for ligne in open(chemin, encoding="utf-8"):
        try:
            d = json.loads(ligne)
            res[d["id"]] = d["texte"]
        except (json.JSONDecodeError, KeyError):
            pass                                      # derniere ligne coupee par un arret brutal
    return res


def teleocr_par_lot(elements, entree):
    """elements : [{id, chemin}]. Renvoie {id: texte}.
    Un cache par entree ET par reglages ; chaque lecture est gardee des qu'elle est faite."""
    jetons, cote = JETONS[entree], COTE_MAX[entree]
    nom = f"{entree}_t{jetons}_c{cote or 'orig'}"
    cache = DOSSIER_TELE / f"{nom}.jsonl"

    # reprise d'anciens caches faits avec les memes reglages
    if not cache.exists() and ANCIENS_REGLAGES.get(entree) == (jetons, cote):
        anciens = _lire_jsonl(DOSSIER_TELE / f"{entree}.jsonl") | _lire_jsonl(DOSSIER_TELE / f"{entree}_partiel.jsonl")
        if anciens:
            with open(cache, "w", encoding="utf-8") as f:
                for i, t in anciens.items():
                    f.write(json.dumps({"id": i, "texte": t}, ensure_ascii=False) + "\n")
            print(f"  {entree} : {len(anciens)} lectures reprises de l'ancien cache")

    deja = _lire_jsonl(cache)
    a_faire = [e for e in elements if e["id"] not in deja]
    print(f"  {entree} ({jetons} jetons, image {cote or 'originale'}) : {len(deja)} deja lues, "
          f"{len(a_faire)} a lire", flush=True)
    if not a_faire:
        return {e["id"]: deja[e["id"]] for e in elements if e["id"] in deja}

    subprocess.run(["pkill", "-f", "lire_lot.py"], capture_output=True)    # pas deux TeleOCR sur le GPU
    man = DOSSIER_TELE / f"{nom}_manifeste.jsonl"
    with open(man, "w", encoding="utf-8") as f:
        for e in a_faire:
            f.write(json.dumps({"id": e["id"], "chemin": str(e["chemin"]), "consigne": CONSIGNE_TELEOCR,
                                "max_tokens": jetons, "cote_max": cote}) + "\n")
    journal_chemin = DOSSIER_TELE / f"{nom}.log"
    journal = open(journal_chemin, "w")
    p = subprocess.Popen([str(PY_TELE), str(DOSSIER_TELE / "lire_lot.py"), str(man), str(cache), REVISION_TELEOCR],
                         stdout=journal, stderr=journal, env={**os.environ, "HF_HOME": "/tmp/hf-cache"})
    t0, n0 = time.time(), len(deja)
    try:
        while p.poll() is None:
            time.sleep(INTERVALLE)
            n = len(_lire_jsonl(cache)) - n0
            if n:
                vitesse = (time.time() - t0) / n
                print(f"  {entree} : {n}/{len(a_faire)} | {vitesse:.1f} s/image | "
                      f"reste ~{vitesse * (len(a_faire) - n) / 60:.0f} min", flush=True)
            else:
                print(f"  {entree} : chargement du modele... ({time.time() - t0:.0f} s)", flush=True)
    except KeyboardInterrupt:
        print(f"  {entree} : interrompu - les lectures faites sont gardees, relancer pour reprendre")
        raise
    finally:
        if p.poll() is None:
            p.terminate()
        journal.close()

    deja = _lire_jsonl(cache)
    echecs = [l for l in open(journal_chemin, encoding="utf-8", errors="ignore") if l.startswith(("ECHEC", "POIDS"))]
    if echecs:
        print(f"  {entree} : {len(echecs)} message(s) d'echec, voir {journal_chemin}")
    if p.returncode not in (0, -15) and len(deja) == n0:
        raise RuntimeError(f"TeleOCR n'a rien lu : voir {journal_chemin}")
    return {e["id"]: deja[e["id"]] for e in elements if e["id"] in deja}


# --- Toutes les lectures TeleOCR ---------------------------------------------------------
CLES_T = sorted({p.stem.split("__")[0] for p in (IMAGES / "ligne").glob("*.png")})
CLES_T = CLES_T[:N_TELE] if N_TELE else CLES_T
FICHIERS = {
    "ligne": [(f"ligne|{k}|{c}", IMAGES / "ligne" / f"{k}__{c}.png") for k in CLES_T for c in ZONES],
    "bloc": [(f"bloc|{k}|", IMAGES / "bloc" / f"{k}.png") for k in CLES_T],
    "page": [(f"page|{k}|", IMAGES / "page" / f"{k}.jpg") for k in CLES_T],
}
t0 = time.time()
TELE = {}
for entree in ENTREES_TELE:
    elements = [{"id": i, "chemin": c} for i, c in FICHIERS[entree] if c.exists()]
    TELE |= teleocr_par_lot(elements, entree)
DUREE_TELE = time.time() - t0
print(f"\nTeleOCR (revision {REVISION_TELEOCR[:8]}) : {len(TELE)} lectures | {len(CLES_T)} bulletins | "
      f"entrees {ENTREES_TELE} | {DUREE_TELE / 60:.1f} min | service : {MODELES_SERVICE}")


# =====================================================================================
# Une lecture, quel que soit le modele et l'entree
# =====================================================================================
def lire(modele, entree, cle):
    """Une lecture -> ({champ: texte}, texte brut, secondes).
    Pour TeleOCR, la lecture vient du cache calcule par lots ci-dessus (temps reel : DUREE_TELE)."""
    t0 = time.time()
    if entree == "ligne":
        res, brut = {}, []
        for c in ZONES:
            if modele == "teleocr":
                sortie = TELE.get(f"ligne|{cle}|{c}", "")
            else:
                im = Image.open(IMAGES / "ligne" / f"{cle}__{c}.png")
                sortie = service(modele, im, PROMPT_LIGNE.format(champ=NOM_CHAMP[c]), max_jetons=60)
            res[c] = sortie
            brut.append(sortie)
        return res, " || ".join(brut), (time.time() - t0) / len(ZONES)
    if modele == "teleocr":
        sortie = TELE.get(f"{entree}|{cle}|", "")
        return bh.extraire_champs(sortie), sortie, time.time() - t0
    dossier, ext = ("bloc", "png") if entree == "bloc" else ("page", "jpg")
    im = Image.open(IMAGES / dossier / f"{cle}.{ext}")
    sortie = service(modele, im, PROMPT_BLOC if entree == "bloc" else PROMPT_PAGE,
                     cote_max=None if entree == "bloc" else 2000)
    return json_champs(sortie), sortie, time.time() - t0

  ligne (64 jetons, image originale) : 338 deja lues, 0 a lire
  bloc (256 jetons, image originale) : 169 deja lues, 0 a lire

TeleOCR (revision e9258535) : 507 lectures | 169 bulletins | entrees ['ligne', 'bloc'] | 0.0 min | service : ['gemma4-26b-moe', 'qwen3-vl', 'chandra-ocr-2']


In [8]:
import re
import unicodedata


def _majuscules(t):
    return unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().upper()


def extraire_champs(texte):
    """Nom et prenom retrouves dans le texte d'un bloc ou d'une page, grace aux libelles imprimes."""
    plat = "\n".join(_majuscules(l) for l in str(texte).splitlines())
    res = {}
    for champ, libelle in [("PRENOM", r"\bPRENOMS?\b"), ("NOM", r"\bNOM\b")]:
        m = (re.search(libelle + r"[^:\n]{0,25}:\s*([^\n]*)", plat)
             or re.search(libelle + r"\s+([^\n]*)", plat))
        valeur = m.group(1) if m else ""
        valeur = re.split(r"\bPRENOMS?\b|\bNOM\b|\bSEXE\b|\bNE\b|\bDATE\b", valeur)[0]
        res[champ] = normaliser(valeur)
    return res


if not hasattr(bh, "extraire_champs"):
    bh.extraire_champs = extraire_champs
print("extraire_champs disponible")

extraire_champs disponible


In [9]:
# 3. Essai sur un bulletin : regarder les sorties brutes avant de tout lancer
cle0 = APP.cle.iloc[0]
for modele in MODELES:
    for entree in ENTREES:
        res, brut_, s = lire(modele, entree, cle0)
        print(f"{modele:15} {entree:6} {s:5.1f} s | {repr(brut_[:140])}")
print("\nVerifier : des noms et prenoms lisibles, pas des messages d'erreur.")

teleocr         ligne    0.0 s | 'Theiler || 0'
teleocr         bloc     0.0 s | 'Exemple : DUPAS, épouse MAURIN\nNom : PÉRÉZ Seau\nPrénom :'
gemma4-26b-moe  ligne    0.3 s | 'PEREZ JEAN || ALEXANDRE'
gemma4-26b-moe  bloc     0.4 s | '```json\n{"NOM": "PEREZ JEAN", "PRENOM": "ILLISIBLE"}\n```'
qwen3-vl        ligne    0.3 s | 'PEREZ JEAN || ILLISIBLE'
qwen3-vl        bloc     0.5 s | '{"NOM": "PEREZ", "PRENOM": "Jean"}'
chandra-ocr-2   ligne    0.5 s | 'FEREL-DEAN || A. B. C. D. E. F. G. H. I. J. K. L. M. N. O. P. Q. R. S. T. U. V. W. X. Y. Z.'
chandra-ocr-2   bloc     0.4 s | '[{"NOM": "PEREZ Jean", "PRENOM": "Jean"}]'

Verifier : des noms et prenoms lisibles, pas des messages d'erreur.


In [10]:
# 4. Toutes les lectures (mises en cache : une relance reprend ou elle s'etait arretee)
cache = RESULTATS / f"teleocr_vs_multimodaux_lectures{SUFFIXE}.csv"
LU = pd.read_csv(cache, dtype=str).fillna("") if cache.exists() else pd.DataFrame(
    columns=["modele", "entree", "cle", "champ", "lu", "brut", "secondes"])
fait = set(zip(LU.modele, LU.entree, LU.cle))
rows, t_debut = [], time.time()
for k, cle in enumerate(APP.cle):
    for modele in MODELES:
        for entree in ENTREES:
            if (modele, entree, cle) in fait:
                continue
            res, brut_, s = lire(modele, entree, cle)
            for c in ZONES:
                rows.append({"modele": modele, "entree": entree, "cle": cle, "champ": c,
                             "lu": res.get(c, ""), "brut": brut_[:500], "secondes": round(s, 2)})
    if rows and ((k + 1) % 10 == 0 or k + 1 == len(APP)):
        LU = pd.concat([LU, pd.DataFrame(rows)], ignore_index=True)
        LU.to_csv(cache, index=False)
        rows = []
        print(f"  {k + 1}/{len(APP)} bulletins — {(time.time() - t_debut) / 60:.1f} min")
print(f"{len(LU)} lectures")

  10/169 bulletins — 0.5 min
  20/169 bulletins — 1.1 min
  30/169 bulletins — 1.7 min
  40/169 bulletins — 2.3 min
  50/169 bulletins — 2.9 min
  60/169 bulletins — 3.5 min
  70/169 bulletins — 4.1 min
  80/169 bulletins — 4.7 min
  90/169 bulletins — 5.4 min
  100/169 bulletins — 6.0 min
  110/169 bulletins — 6.6 min
  120/169 bulletins — 7.2 min
  130/169 bulletins — 7.9 min
  140/169 bulletins — 8.5 min
  150/169 bulletins — 9.1 min
  160/169 bulletins — 9.9 min
  169/169 bulletins — 10.6 min
2704 lectures


In [11]:
# 5. Le resultat : justes, avouees, silencieuses, CER, temps — et McNemar entre les deux modeles
LU["lu_n"] = LU.lu.map(normaliser)
LU["attendu"] = [REF.get((c, ch), "") for c, ch in zip(LU.cle, LU.champ)]
LU = LU[LU.attendu != ""].copy()
LU["exact"] = (LU.lu_n == LU.attendu).astype(int)
LU["avoue"] = LU.lu_n.isin(["", "ILLISIBLE"]).astype(int)
LU["silencieux"] = ((LU.exact == 0) & (LU.avoue == 0)).astype(int)
LU["dist"] = [bh.distance(a, b) for a, b in zip(LU.lu_n, LU.attendu)]
LU["len"] = LU.attendu.str.len().clip(lower=1)

lignes = []
for entree in ENTREES:
    paire = LU[LU.entree == entree].pivot_table(index=["cle", "champ"], columns="modele",
                                                values="exact", aggfunc="first")
    for modele in MODELES:
        g_ = LU[(LU.entree == entree) & (LU.modele == modele)]
        if not len(g_):
            continue
        lo, hi = bh.wilson(int(g_.exact.sum()), len(g_))
        ligne = {"entree": entree, "modele": modele, "N": len(g_),
                 "exact_%": round(g_.exact.mean() * 100, 1),
                 "IC95": f"[{lo * 100:.1f} ; {hi * 100:.1f}]",
                 "avoue_%": round(g_.avoue.mean() * 100, 1),
                 "silencieux_%": round(g_.silencieux.mean() * 100, 1),
                 "CER_%": round(g_.dist.sum() / g_["len"].sum() * 100, 1),
                 "s_par_appel": round(g_.secondes.astype(float).mean(), 2),
                 "McNemar_p_vs_teleocr": float("nan")}
        if modele != "teleocr" and {"teleocr", modele} <= set(paire.columns):
            pp = paire[["teleocr", modele]].dropna()
            n10 = int(((pp["teleocr"] == 1) & (pp[modele] == 0)).sum())
            n01 = int(((pp["teleocr"] == 0) & (pp[modele] == 1)).sum())
            ligne["McNemar_p_vs_teleocr"] = round(stats.binomtest(n10, n10 + n01).pvalue
                                                  if n10 + n01 else 1.0, 4)
        lignes.append(ligne)
RESULTAT = pd.DataFrame(lignes)
RESULTAT.to_csv(RESULTATS / f"teleocr_vs_multimodaux{SUFFIXE}.csv", index=False)
with pd.option_context("display.max_columns", 20):
    display(RESULTAT)
print("\nexact : justes | avoue : 'illisible' ou vide | silencieux : faux sans le signaler"
      "\nMcNemar_p_vs_teleocr < 0,05 : l'ecart avec TeleOCR est reel, sur les memes lignes")

,entree,modele,N,exact_%,IC95,avoue_%,silencieux_%,CER_%,s_par_appel,McNemar_p_vs_teleocr
0,ligne,teleocr,335,23.3,[19.1 ; 28.1],3.3,73.4,48.0,0.00,NaN
1,ligne,gemma4-26b-moe,335,63.3,[58.0 ; 68.3],1.8,34.9,13.6,0.36,0.0
2,ligne,qwen3-vl,335,54.3,[49.0 ; 59.6],15.5,30.1,32.4,0.33,0.0
3,ligne,chandra-ocr-2,335,46.3,[41.0 ; 51.6],0.0,53.7,55.3,0.34,0.0
4,bloc,teleocr,335,38.8,[33.7 ; 44.1],24.5,36.7,37.8,0.00,NaN
5,bloc,gemma4-26b-moe,335,84.8,[80.5 ; 88.2],0.6,14.6,6.0,0.60,0.0
6,bloc,qwen3-vl,335,78.2,[73.5 ; 82.3],0.0,21.8,7.1,0.60,0.0
7,bloc,chandra-ocr-2,335,72.8,[67.8 ; 77.3],6.0,21.2,13.2,0.49,0.0



exact : justes | avoue : 'illisible' ou vide | silencieux : faux sans le signaler
McNemar_p_vs_teleocr < 0,05 : l'ecart avec TeleOCR est reel, sur les memes lignes


In [14]:
suivi.ETUDES.setdefault("banc-htr", "banc des modeles de lecture : PyLaia, TrOCR, TeleOCR et multimodaux, memes lignes")
suivi.ETUDES.setdefault("banc-decoupes", "methode de decoupe des champs x lecteurs ajustes")

'methode de decoupe des champs x lecteurs ajustes'

In [15]:
# 6. Trace MLflow (mode complet seulement)
DEPOT_T = f"XingChen-AGI/TeleOCR@{REVISION_TELEOCR[:8]}"
COLONNES_NOMINATIVES = {"lu", "brut", "attendu", "texte", "lu_n", "attendu_n"}

if TEST_RAPIDE:
    print("Mode TEST RAPIDE : rien n'est enregistre. Passer TEST_RAPIDE = False et relancer 1 a 6.")
else:
    # Artefact : uniquement le tableau de synthese (taux, CER, temps), jamais de lectures ligne par ligne
    f_synthese = RESULTATS / "teleocr_vs_multimodaux_synthese.csv"
    RESULTAT.to_csv(f_synthese, index=False)
    nominatives = COLONNES_NOMINATIVES & set(RESULTAT.columns)
    assert not nominatives, f"le tableau de synthese contient des colonnes nominatives : {nominatives}"

    with suivi.essai("banc-htr", notebook="08", nom="teleocr-vs-multimodaux", modele="teleocr",
                     depot=DEPOT_T, phase="lecture", comparaison=",".join(MODELES_SERVICE)) as run:
        mlflow.log_params({"bulletins": len(APP), "entrees": ",".join(ENTREES),
                           "modeles": ",".join(MODELES), "teleocr": DEPOT_T,
                           "teleocr_revision": REVISION_TELEOCR,
                           "teleocr_environnement": "transformers 4.x, processus separe, lecture par lots"})

        # Temps reel de TeleOCR (ses lectures viennent du cache : s_par_appel vaut ~0 et n'est pas comparable)
        n_tele = sum(1 for k in TELE if k.split("|")[0] in ENTREES)
        if n_tele and globals().get("DUREE_TELE", 0) > 1:
            mlflow.log_metric("teleocr_s_par_lecture_reel", DUREE_TELE / n_tele)
        else:
            print("temps reel de TeleOCR non mesure dans cette session (lectures relues depuis le cache)")

        for _, r in RESULTAT.iterrows():
            nom = r["modele"].replace("-", "_")
            for k in ["exact_%", "avoue_%", "silencieux_%", "CER_%", "s_par_appel", "McNemar_p_vs_teleocr"]:
                if k in r and r[k] == r[k]:
                    if k == "s_par_appel" and r["modele"] == "teleocr":
                        continue                    # temps du cache, trompeur
                    mlflow.log_metric(f"{nom}_{r['entree']}_{k.replace('%', 'pct')}", float(r[k]))
        mlflow.log_artifact(str(f_synthese))
    print("essai enregistre :", run.info.run_id[:8])

temps reel de TeleOCR non mesure dans cette session (lectures relues depuis le cache)
🏃 View run teleocr-vs-multimodaux at: https://user-farisazibert-mlflow.user.lab.sspcloud.fr/#/experiments/1/runs/a55dff76c4ba422a89492309320a670a
🧪 View experiment at: https://user-farisazibert-mlflow.user.lab.sspcloud.fr/#/experiments/1
essai enregistre : a55dff76
